### Pipeline de base pour le clusturing des clients a tester sur les données optimiser par le rfm



In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
print("Dependances de bases pret..")

In [ ]:
df= pd.read_csv("../data/processed/rfm_data_processed.csv", sep=",")
print(f"Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(df.head())

In [ ]:
df.info()

### construction d'un pipeline de base pour le test clustering sur les données nétoyée


### Preparation des données

In [ ]:
# 1. Nettoyage des données
#    - Les vraies données de vente contiennent souvent des retours
#      (Quantity négative) et des annulations (Invoice commençant par 'C')
#      qu'il faut retirer avant de calculer le RFM.

# df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
 
# Retirer les annulations (Invoice commençant par 'C') si présentes
# df = df[~df["Invoice"].astype(str).str.startswith("C")]
 
# Retirer les quantités ou prix négatifs/nuls (retours, erreurs)
# df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
 
# Retirer les lignes sans CustomerID (impossible de rattacher à un client)
# df = df.dropna(subset=["CustomerID"])
 
# Calcul du montant par ligne de commande
# df["MontantLigne"] = df["Quantity"] * df["Price"]
 
# print(f"\nAprès nettoyage : {df.shape[0]} lignes, {df['CustomerID'].nunique()} clients uniques")
 

In [ ]:

# 2. Calcul du RFM par client

# date_reference = df["InvoiceDate"].max() + pd.Timedelta(days=1)  # jour suivant la dernière transaction
 
# rfm = df.groupby("CustomerID").agg(
#     recence_jours=("InvoiceDate", lambda x: (date_reference - x.max()).days),
#     frequence=("Invoice", "nunique"),          # nombre de commandes distinctes
#     montant_total=("MontantLigne", "sum")
# ).reset_index()
 
# print("\nAperçu du RFM calculé :")
# print(rfm.head())
# print(rfm[["recence_jours", "frequence", "montant_total"]].describe())
 

In [ ]:
# 3. Normalisation
#    On applique aussi un log sur montant et fréquence, car ces
#    variables sont souvent très asymétriques (quelques gros clients
#    achètent énormément) -> le log réduit l'effet des valeurs extrêmes.

rfm_log = df.copy()
rfm_log["Frequency"] = np.log1p(rfm_log["Frequency"])
rfm_log["Monetary"] = np.log1p(rfm_log["Monetary"])
 
features = rfm_log[["Recency", "Frequency", "Monetary"]]
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)
 

In [ ]:

# 4. Méthode du coude pour choisir k
inertias = []
k_range = range(1, 10)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(features_scaled)
    inertias.append(km.inertia_)
 
plt.figure(figsize=(6, 4))
plt.plot(list(k_range), inertias, marker="o")
plt.xlabel("Nombre de clusters (k)")
plt.ylabel("Inertie")
plt.title("Méthode du coude")
plt.tight_layout()
plt.savefig("../figures/methode_coude.png")
plt.close()
 

In [ ]:
# 5. Clustering final
#    -> Ajuste k_final selon le coude obtenu (souvent 4 à 5 pour du RFM)
k_final = 4
kmeans = KMeans(n_clusters=k_final, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(features_scaled)
 

In [ ]:

# 6. Interprétation des clusters (sur les valeurs réelles, pas le log)
resume = df.groupby("cluster")[["Recency", "Frequency", "Monetary"]].mean().round(1)
resume["nb_clients"] = df.groupby("cluster").size()
resume["% clients"] = (resume["nb_clients"] / len(df) * 100).round(1)
 
med_r, med_f, med_m = resume["Recency"].median(), resume["Frequency"].median(), resume["Monetary"].median()
 
def etiqueter(row):
    if row["Recency"] <= med_r and row["Monetary"] >= med_m:
        return "Clients VIP / fidèles"
    elif row["Recency"] > med_r and row["Frequency"] <= med_f:
        return "Clients perdus / inactifs"
    elif row["Frequency"] <= med_f:
        return "Nouveaux clients prometteurs"
    else:
        return "Clients réguliers"
 
resume["profil"] = resume.apply(etiqueter, axis=1)
print("\nProfil moyen et interprétation par cluster :")
print(resume)
 

In [ ]:
# 7. Visualisation

plt.figure(figsize=(7, 5))
scatter = plt.scatter(df["Frequency"], df["Monetary"], c=df["cluster"], cmap="viridis", alpha=0.6)
plt.xlabel("Fréquence d'achat (nb commandes)")
plt.ylabel("Montant total dépensé (£)")
plt.title("Segmentation clients (K-means sur RFM)")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig("../figures/clusters_visualisation.png")
plt.close()
 

In [ ]:
# 8. Export
rfm_final = df.merge(resume[["profil"]], on="cluster")
rfm_final.to_csv("../outputs/customers_segmented.csv", index=False)
print("\nFichiers générés : methode_coude.png, clusters_visualisation.png, customers_segmented.csv")
 